# Released WHAM vs the proposed iPhone subset on 3DPW

This is the final accuracy tradeoff run. It evaluates the official and controlled rows on every 3DPW test person track, then evaluates the app-like row against a WHAM reference on every single-person test video:

1. released WHAM with the official flip average;
2. released WHAM without flip (the fair controlled reference);
3. phase-three FastViT with official keypoints and initialization; and
4. a same-population WHAM reference for the single-person videos; and
5. the actual proposed iPhone subset on those videos: YOLOv8n-pose + phase-three FastViT + neutral/identity initialization.

The current app selects one highest-confidence person and has no multi-person identity association. Restricting only the app comparison to single-person videos prevents one detection from being incorrectly scored against two different people.

The notebook uses licensed SMPL files only to decode predictions for the paper's PA-MPJPE, MPJPE, PVE, and acceleration metrics. SMPL is **not** silently added to the phone workload. WHAM's official 3DPW evaluator sets camera angular velocity to zero, so this is camera-coordinate body reconstruction, not a world-grounded trajectory test.

Required Kaggle inputs:

- your existing private `3dpw-model` dataset with `imageFiles/`, `sequenceFiles/`, and `3dpw_test_vit.pth`;
- the saved phase-three notebook output containing the exact epoch-36 `fastvit_hmr2_best.pth`; and
- one private SMPL-assets dataset containing the three licensed SMPL model files. The next cell searches all Kaggle inputs, so the dataset slug does not matter. WHAM's public `J_regressor_h36m.npy` is fetched separately with a pinned checksum.


In [ ]:
# Configuration. Usually no edit is needed.
from pathlib import Path
import hashlib

KAGGLE_INPUT = Path('/kaggle/input')
THREEDPW_ROOT = Path('/kaggle/input/datasets/nguyntrunglong/3dpw-model')
SAVED_NOTEBOOKS_ROOT = Path('/kaggle/input/notebooks/nguyntrunglong')
SCRATCH_DIR = Path('/tmp/wham_full_pipeline_tradeoff')
OUTPUT_DIR = Path('/kaggle/working/wham_full_pipeline_tradeoff')
SEQUENCES = 0  # 0 = all matching test tracks.
FRAMES_PER_SEQUENCE = 0  # 0 = every frame.
POSE_BATCH_SIZE = 32
STUDENT_BATCH_SIZE = 64
SMPL_BATCH_SIZE = 256  # lower to 128 if the GPU runs out of memory.

EXPECTED_STUDENT_SHA256 = (
    'f15875f3fed12538312f59956b6c93e9cca2ab41a9e8d87edf593dd85f311ab1'
)

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

checkpoint_candidates = sorted(
    SAVED_NOTEBOOKS_ROOT.rglob('fastvit_hmr2_best.pth')
)
phase3_matches = [
    path for path in checkpoint_candidates
    if sha256_file(path) == EXPECTED_STUDENT_SHA256
]
if len(phase3_matches) != 1:
    raise FileNotFoundError(
        'Attach the saved epoch-36 phase-three notebook output. '
        f'Expected SHA-256 {EXPECTED_STUDENT_SHA256}; found '
        f'{[(str(path), sha256_file(path)) for path in checkpoint_candidates]}.'
    )
STUDENT_CHECKPOINT = phase3_matches[0]
if not THREEDPW_ROOT.is_dir():
    raise FileNotFoundError(
        f'Change THREEDPW_ROOT to the existing private 3DPW mount: {THREEDPW_ROOT}'
    )
parsed_candidates = list(THREEDPW_ROOT.rglob('3dpw_test_vit.pth'))
if not parsed_candidates:
    parsed_candidates = list(KAGGLE_INPUT.rglob('3dpw_test_vit.pth'))
if len(parsed_candidates) != 1:
    raise FileNotFoundError(
        f'Expected exactly one 3dpw_test_vit.pth, found {parsed_candidates}'
    )
PARSED_3DPW = parsed_candidates[0]
print(f'Phase-three checkpoint: {STUDENT_CHECKPOINT}')
print(f'3DPW root: {THREEDPW_ROOT}')
print(f'Parsed labels: {PARSED_3DPW}')
print(f'Useful outputs only: {OUTPUT_DIR}')


In [ ]:
# Runtime dependencies. Resolver warnings about unrelated preinstalled Kaggle packages are harmless.
%pip install -q timm==1.0.22 ultralytics==8.4.146 coremltools==9.0 einops==0.8.1 yacs==0.1.8 joblib==1.5.2 loguru==0.7.3 smplx==0.1.28 chumpy==0.70 opencv-python-headless==4.10.0.84 scikit-image==0.25.2 tqdm==4.67.1


In [ ]:
# Locate and normalize the three manually licensed SMPL model files.
import shutil, urllib.request

SCRATCH_DIR.mkdir(parents=True, exist_ok=True)
SMPL_MODEL_DIR = SCRATCH_DIR / 'smpl'
SMPL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_aliases = {
    'SMPL_NEUTRAL.pkl': [
        'SMPL_NEUTRAL.pkl', 'basicModel_neutral_lbs_10_207_0_v1.0.0.pkl'
    ],
    'SMPL_MALE.pkl': [
        'SMPL_MALE.pkl', 'basicmodel_m_lbs_10_207_0_v1.0.0.pkl',
        'basicModel_m_lbs_10_207_0_v1.0.0.pkl'
    ],
    'SMPL_FEMALE.pkl': [
        'SMPL_FEMALE.pkl', 'basicModel_f_lbs_10_207_0_v1.0.0.pkl'
    ],
}

def find_asset(names):
    matches = []
    for name in names:
        matches.extend(KAGGLE_INPUT.rglob(name))
    matches = sorted(set(matches))
    if not matches:
        raise FileNotFoundError(
            f'Missing licensed asset {names}. Upload it in a private Kaggle dataset.'
        )
    if len(matches) > 1:
        print(f'Multiple matches for {names}; using {matches[0]}')
    return matches[0]

for output_name, aliases in model_aliases.items():
    source = find_asset(aliases)
    shutil.copy2(source, SMPL_MODEL_DIR / output_name)
    print(f'{output_name}: {source}')
H36M_REGRESSOR = SCRATCH_DIR / 'J_regressor_h36m.npy'
H36M_REGRESSOR_URL = (
    'https://huggingface.co/camenduru/WHAM/resolve/main/'
    'J_regressor_h36m.npy?download=true'
)
H36M_REGRESSOR_SHA256 = (
    'c655cd7013d7829eb9acbebf0e43f952a3fa0305a53c35880e39192bfb6444a0'
)
if not H36M_REGRESSOR.is_file():
    print('Downloading checksum-pinned public H36M joint regressor...')
    urllib.request.urlretrieve(H36M_REGRESSOR_URL, H36M_REGRESSOR)
actual_regressor_sha256 = sha256_file(H36M_REGRESSOR)
if actual_regressor_sha256 != H36M_REGRESSOR_SHA256:
    raise RuntimeError(
        f'J_regressor_h36m.npy SHA-256 mismatch: {actual_regressor_sha256}'
    )
print(f'J_regressor_h36m.npy: {actual_regressor_sha256}')


In [ ]:
# Materialize the reviewed scripts embedded in this notebook.
import base64, gzip, hashlib

embedded = {
    'evaluate_full_pipeline_tradeoff.py': ('4c09f9d1c407f77b3771764fbd7ca24801419ace0e7b3be2aeac78fc6f9556fb', 'H4sIANJPomoC/9U9YXfjNo7f/StY7YeVO7ISJ53Zrve872Y709fZm2nz2tnuu+fz05NlOlEjS1pJTuLm8t8PIEiJpCjbmWnv3c2HiSWRIAiAAAgC0h++ONvV1dkqzc94fsfKfXNT5Jcjz/M+8LjeVZw1N5xdvrn6J4uTZFfFyZ41VbzmxWbDNlWxZRXPoCVfs39+9/oDawrRIb0CMJzVu1XNm3A0+gj3+F2c7eIGWlbFfc3WPEtXvIIb2Z7VvIzxJ2vuC8Y3G5409Ww0moYCmjlEGTc37D6F/9KmZoBHmqRxxjZZWqox0iL/y+iCOtfxVoPQdi52Tb/LZciKHNDBft99+PGCpdv4mrMNjxskRcXLLE4AyoqarNO6SbMMbnwb183P6UeY/i3P/8LifD36ioYvq6IscGSDJMz/zx/e/3D3NcNngeoesJzvgLoZS/O0gUmlvwrExpKCm7SqG4BacU403NVIVl7tWcmrusiRNcltyLBxXJaTLL0VLREhFrdUzvYjmEpRNYqkFd/wiucJ1yDWaX6d8YkEfJcCywO24kmMTXBmIA3Qp8GBRuuC1ywv4KKuC2AHcBI65A3MAh6kOdvusiY1gIXsdZbRNGKg7ZonxRrwQd6MsjThORLtpw9X7xEmB05/uSkqjVuCU1+KmdVJUcmuHcuvXk8+XP396m0woj/s6mf4TxAiSUAcKoKCo+KsBVORGH+sSdzlSEVF5CT2jWCygBx1bSedFHkTwySbm7QmlDU0aWKhAPpHTV5xrFE7CKJQs195VbAEsK9iwPR6l8UVu+MZULTZB6wuYDQQoWJDqwJJt4Kn9wyGjUf3RZWtJ9dVscuRkCAKv8AyKoCVHTYhruzRSCzcKNrsUKyjCMQcpQGGhNmIdvVopO5V1yA0NVfXSX2nfqZ5XcII6vIXYK36Xe9r9bNJt5wGTApYK4kAr0b8BpBteBUAlTYxiMg6BXiiMS5UUBCq4RVc0oNmX4Jkqvuv832L6i/FCnqoq3y3LfcgOywvW1SKKlFQ/rXeKhj4m+4CBrD69k2atBjiOm1HUCos2harNONRmZagxXIeXa7LexxL3sd2vT73N/E2kqokQjUAq2MnRAQ6qvui52j08+sf373+/uNPbM78EYN/XhnD4iEYK77OEBRoLy8YepoXRoMNaJi7tImU+EVpXu6aWj1OS5TuiJRTtC+y4u5reDYefXf56kP08Yfo79OvAJfFq4C9DNhXAZsG7CJgl/ADbk3h3hRvwt0p3J7C/a/hz/lydPX2/c/vforeff/m3TdvcToL7LYcfXuFF5fnMFdgPXI7uYkyfg32Jaq3ZRateclBjvMEFIg/ZpO/su8Bwxmh63mgO0DuTT1xN2VlmtxmsJDADGVFvAYdASxZ8ypnV8KynX2/217tQ7EIEFK6EQv4Jq7jpql8KdEB8655A5KPV96YBhXN6XnYPYVJaDc3uyxTD9gfUFT5jKXXOainBQ4wgbmCuKyXAiLo9xhX/Zw9tiN4q6LIvBkIbYi/oqB7kuYNPID/tXsbmCXeFX+1+0kBNOQP8ET+0p4VK9QL8Ih+aE/qpoLb8L92b5enqL/0+0/if9TGOaiqgKGAc1TyckJh2vAtME2j20Y0FbSGdjC5KMKlHkVdG/wHwif4kJeBDnssGjXVvmsdRbS8osj3khtc6h614g8JLxv2Tjx9W1VF1XX6A8lJmJe/gi1FOVkXAqecgwwRmJCx1ywBb6HiYG2wP9gVMFQrULhxSnZCA4iy+oDzA+u6Q9sADUiMWVjeZmBrml2cgUuBQ4BSAWdIdS7Brknxr/d5clMVefor99f8DqR6RsoqpCtL/mE4uh+ihLH5HBi+W8deN1PqjDdDHfZYjoc6ee0cKWBFKU3jDJWrGLnZgQgt4CogQVvSQH2siQU1LA50LeZinBDgbaKEFL1PLSpeg6qFBu1g8sEQyIqDdsxlv8AFl03UuHKOYJTBTvMcVW+9227jau8LaapnoDfqZgFSmK/jqor3SzFLFMgFCjlNkv03rrXlTNcTsn9LZiER7EdAARASwuaD39xUaSK8ZXB6hG0H88y3ZaNEVE5GV/khYYjTBrQ01CXKY8W5+C5Os3gFNmZTwQKpfbjgGUypwx74tAwA9zV/EOpCTA7+EtqiG2ja/Lq5QdWzaCcD9yS0xS3fLxcCApKGTds2uOjhIS5i31i53m15sfYC895qVTzY9+Ss6959MFYlX0cuOOqZE558NgQXHeweXqBm+ggIwqT64MSupc60bfzgb9PcN8g4Dti5YpBlYIlKA0wi0WoZFXQMqrUbJXjn6YOmf53rdmTJMD38COaxqKQQA+PKAsAi2w3ha4pIeEuK/TQieyG52oqCgIJMOB2AYJkJAJhUZOT4HAHjIT9V58W57P8QgUINgBP1Lf6ygYCx3eLOCWRC+EOCD52wtkQIxEwCDZ1Rx3Nib43+BcAXf31wbog3MK/pWPoPsE8Dkb1c21hURdFE4JyAbgIdlRQdAicSroXcUQ8ooKEoG8DAlsbw264hzO0GPEN/MgURFXRDKsIu6CGt5+caHFwlEd6NYtz1oeoW8oNescTwkyYgVp8+gRD8ULBZpvKACQiVe3lhLb1xO4MLcC8vLYRtmm9j0LwPEaBUyW1M9Go9gHc3V2xPPX0XKWxEpuT7AjqvpAygKwhOs9icR8j3T6eeN0ivjkRuyijZEFOQSPy2xOlN002ZV4aN0xxbdEZ7ZCGZ1Fbbchzu8vpfO87BFJ6PgULKD9B8Ulz6LmhKJTwbYGs8Zgc49wzZb+F1rDTAWCrGeDbI8m5Z4D9jVt1d5/SkQnDNTj5yjKkBNTn89dcnjSdEedat1gNNUZpUU/ztbpqAMgehvONZOxGMl9SdYgbpC6R1nBv9n6RxrnZ5hLEiYmrOm/uiulXA8jz8UKx3GVd2Ge33bMikSmMtGT0zHoLriM46LEH8c4ppdrmEiKywYB3GGtZz+bcj0MOcUF7AUlt2t3FRtE/EytEeKvzbBp3kos+rrtCDFTMCMW/DoXWg7YuFFLVQlMAtrSYoB2YjUnhWM5QBs5mQEK1ZJwttO008ZMPOZ4ZNxjWPlKsI6xSZgbsO0U4QGQOiMxFmos7kRhxjP0E80Oo464HM8MhvkRirXXK9r0OMgnXbDXUnTPOaV41/HlhdlfNSbGGLswrBq8lgN74Vf9sg3921oECN29dyn+HOACPZuKmYU5tJARRN13zSFLDxA4qONJIo00tXymSFmortvI7WUGrEak03XT4DgOIjxczVlWgK8Np5aUulQziwbi480c8YV1sWGqq9u8Ndh0yfVIU6/mqOUVz7JiV1lUcDzcy5urpKjCx1hyEwiqZhECyTlBG/o3VaUXRYCr1zV+EW31ZRSgk+FrzrhFJGS0gQRSzmuTIoN+Jg5Ds2+9aM2BnbeAg9erxGNKpwB3yr/PEThmS8cQjeDYZ//LGxr6W2YmsLejKDXSMacvVLnszILfz4YAzA3FgKTL5/+4+PP75+jwgEYubRh9fv39IlnkmIW9++VTfFkYhnuh6wG/rXLsWDDkDx0Zrzk2f5CMTx2QHmYdyxC+WdOPtu3oQA0hEAPY8DJox6QW0QIexjko8go9abo67rRh2b+3eCMac/5qOk4hgEggnkdTb/Ns70RT0O0dj6umNo7PcFhrCe/p3Il+bypEwsKZAgEb4TS0zEfeQa60zMEQsivWpr02G6E0ZLEbtwPaeZa2GC4xJA7X7B7TDYj2vQKwDNBdsdcRC0vdnlt1EN220ZsdACht3AWqjNDCFKOmlWwdxEtzsVbSPSBVBbIrdGpG85yJ/11WbF4nI3vqArIKADVXpehzo9b0HawDRe2jNx7qi6aK1TCHqTuTwyGV1CcA+qXZ46BVCvGAY02TJmX8zFbQ3PZyjAjXfV0vSMQDAKnLFtWm/ReMzYo2PYJ6EZbS34aGNiqD5pkEBhVI0j0IsBT9oob8tfwIQebVbendKoxZvWkt7BWPNdFzmBk9sfj6lLFU2mXqlk8IuV6jbGtRpJBdwaAzEa2gJQmdcc/UwHewJt8WviUIPBRshp7hOYF1o7J6CxsaIjMPzg0aOHqc/IlKlVsd7TbsKEtRBDzhAHkPPZ0rIRWbECeEWV8rw51HM2tXqKRTS3tIXWxWqOAC9Ae/WMjb3k27nqfBmaqib0z5nnULeBSepq43Nm2BELnNcGNIzSrnLSobpt00RrblCp38Fad3UbeAOlst1ZdLTMXOBAcJhPbvg9kNYMpDqyDB3P7lLN0rTYL2YBM4/Ll+GWx7k5jXW6nYMOv+W8xJ8fqx0fRrsdy5jGbzmQgwW9W5Pe7IfJa1xPzHkcFy3r5vGB+7KmdTYH7/YKanstLDv+FxkbbP85mxpj/Yl9FB7Y7/DMMt2mWVylzZ48V9DL20iIX2DbO5lMcJ0LA3EqGJPdNtcCkxcu1ksrG8Yl7vZMcH4v3kkrB6gSZ9ewlHGfEOH21VeYT8wRAyGBk+m4B4mEdfBpUu58x22Kd/YffAn+0Pl5eG48cEVcNY/hc6fca+lcSgME6XX+P0Oh1ln6nenTX6Pqzv8DGrl8RUUvWwQIC7c1MnsakmJ3Q19Ps7yBaVddlrBnykYH0zN4Fpf1kHeqZWYcNNoJbEic1BkbVHdbZOzcp47Zs7e5IRhj9tc5e9k5sEZ66Jz1jiqlvLoF1e9b9cnFEkhwASLSezadTaZLcJB7Dy5my76MTZhv2fEOtPlAwTXvItCgB1WcBV+Y98e0OMSj6ZjAjUxx97+9+unLL3tHU3hAMEhKIJ7Ig8ENxRrPm+b2GZeMunR0fbSyOeKI1PB2S0ly+sG3MkjWJL3hHppKtzuhLnMOcjfQQcy1y6+8PN+UeJ6ok6Dr8dT9lGvHPLLYxrB9srK+RBpuhZFumZIbvq6ud1tw8a/EE1+PZoKHKlIPsriu570Obyjbtv6OZ+W3qrEWwqahwni9jmLZxfcmE3F7PcFkVy8Q2Y1zih2ryKTwEg+CEMnrEwAwEUc6nwilbnaYXT5JbnhyK9I5PhUSnp1M8OzkswB8Ph6Ucju55+n1DablfiJdtmU2EXvISRuL/VRYN5evthOhOSbtDueT8YKWGDRtJ4ahQpXzrRJRBvrS0e4ndMQt6kQ4wxMMPTghKL1zRM6OQHn11XGuHAFx8fLVQRhkqj+V/mCQWx44QMnjtepa7KMIhPiDQPAAR+pmGi6i1FndMmIz6rEWqfCB+USSMeqWidWAjrmHntLaiOTasB6imEb2RtwBHdc4O2MebNzOKOp1hvfDcu/ph3fbtMbCFwy84XEDnjuMRUhMFA5hRMykgjyPolNZday0VO6GhDbT9kyCuFymisrB2oMdAXPGPMwhCZgX4rx8CUQd7sq0azqp0c+WsEd7stRe0JmSZyVjA8o+saY9HbRObx5xkN6BmZmubcymn+DSzq+DJY6uBgd+8iz3mY4Yiu02xRiZnvNd71ZlVYBGqQE9ITnSm+3wWHjXKYq5V/E7Ml148d3b12+8ZcCS+/XclA5YEPyh6QIdY5DbKi0751Ei8oUV28dCnuibHz58ePfxOUHxtw9YNaAqsCTsxyHITwFwfpev2SO17J33yTQCc0qhSG68A8Y9P6tgEJJkC/n/rQNOlz4loiM2Wgo6nmiovGXwakQSiQd7FE8dwsDq9TfeG3nK9Eiwnjw8K9rVN5p+E/V4zvy2rEAXLeoa0ASEvxGBSqLEom4WWl4ugKLSoRAPzH1bmY1N7Sf6RLd8bxVuWInLdvKzK/F5KOl5KOH5ULKzljDqzMRyQHH2MB7aPe1rO5Hao4ME/c5daoLvZ1l73UToODpe1VhXOGe1KFH0XaSfYK2ITLwbtyuAeh5chRtPeMtrKvJL61bjd4WkKc/WoIUfCdqTJwVenFRJ26DlK635g5FGIO6AcY7vI1HnCJPFdcdhJyqqXdtcT6SMtssD9H1NvM/EItShjMOqLrMUDHoEK2OKKZ9CM4P69PVMdanfFbqHqxW+L0iLS3qI+tGa+mINIZYZxveyDLZzKTuCC1WuvDv2V3beDScCAPIkyGxHp0AKQf38p6hTqg+cy512XcZJewDV9sCahIAG0HaSoEWUJ0bFRBkpWOCX6rlATaNGGS/JsstL5FKLwNKxm9UAKniyMAmIRvGOOlrtiWHQSBY3+trRWGVyX2WH9jnbiZKontWHU/kdVGmYpdfpSuRgPB4TSqH+5Rx0sXPiv3gGsl1sYD6XxSJPhiiauD4jdwYrb1uyS+FU1bZ5YdUoCwkWyNdUs6sqlE23wjMrsQl8TYXfeP8G3Exew+ayyDdYxAxGjgYQFbhgpBNRJ2tDXXFVh4zQ6wagxdeAZ91Q8XMJTJIFtpjC01ZHh73ja+kuB+oHBu5BHQvNY9k8zO+iRv5Rl1tabE2GZAqpE6wj83TIIwjcTnx/POmxkZF2jmq0GHZBRuoIsj3IxjJdv3VbzB3DWM+Igra9tLhBh1Sfo5kcJHcaUS4LMDrPwbUhGS+0OtqAzZauxBtX2YA+UJe0MZDQTtLTFp6ZeandLzuPYbk0lMddXKWxOLrqSrJ97EP7IPkYdYkqUtYWPKxjkFMj6UgDo1XXnTKoaHjKqFj7rIyLTKMwK620RArUHhUWoq+jdZVumoM5IqS32rcjRAO07VEUQPSoR/SB5Z+p6vG62FWJKuSDLuf9JmveqIp5fD7q6XOsXfeVhkRi18nc00y5vvFr3ZH5sCUaGeUJtcirdVYcygpDw38R8i87Gq6AAbAtn6sDvYcBSTb+t7kNBdmX5tpRtHIojxa1da5nf7K4/ekBUBIl9xSijWmeOnctaIkbdBgtZlrB2NJxFiTzkymDHathrdpBg9JdkYPnmdpIz1h+NizVUYNp46eOieRlzYEJgt5URmxmDcgq4izertbxrKu4UOUK5qRdJ2QKIzWsuv7McU0KuQZus6yP5lopNLv8nWM1BsdtqEVskwqHsDXI0j97s2Y10rxRchHaco5W/DuQxoLgD6hTmsjuaM5QeS9WpAgWz6KXqHR4wZq1o4spM9aTNYDuOAR9kiucKTlCpKK55MdBZ9VTMab1rIqKnySTxr2efPbCZr26Hqe26D+2uXK87E3M05lM0kmvGeQ49CaSmb4igmNd1OtJZpbcWzXZA28tmdlc6Q7WurNyckUWQ/guF17r43pL9qJ30mtquxc9PfTCXoKuPKxhJBQF+niYI/dhDVFl2T4iONZyfhYgCyWX0LfwCthHVXeUqQ1apukVupv1cLqd73a4Q1tEzQ8DsyV/YRlbe9Hprx4iveN7fecsqmLpt9axHkg+aTcbgfO5U+sZ2s/9iIKNCHxIMz1LjR1SPf00mLGLyvLVY5H+toAux6Kf8KBXYr6cqnCM3MnIs/2WVw9Uf45ZN1aqzIEiUwM56d489hARBcbtMGBqTlOAekGxJl3PBKAVEFtS+UxAXanuEDtO6C/rzXRBV7nU+OhYd1mOq3eX4Yo9RdaH+39GjW5fg2t8V8ZXXh63vaf5hIZUHVobmkFcuF+WtRTZ9BqyRv9W5br7Ljz8JYthUd/2NFm/flwE42FPBhsinrdWyQxKLZ+HhGE9TkFBGZDfYnDL4jg4PXIVzBx82cKzX7hg+ZjOt1UMvLFiKOHPrsr5DGTpuOV3xVaWAZoRAnmq00WAs+Le2AuBJZcd5SmjrD0zhnMexvwjx9dm0asoRdhCwpF1g19UTyLe8ai21+1pDAXk5Q4dX4XoDnap6M/jkxEPl/GkQFvYIvzfLfP+S8yEOqIiv4BK/joN1K8AdGuPvsrURHnwoeB7/6mr1rELdPbvD2ZJHPMauvjokN8xdq51SWVY3KJz9xrKiF6KKX1LjZSmwBT3PU/SafnlOGByFF/7bZSsYCMVqnEoNEo2UsvqkGVCMSKOR9qr6uhlpyQIbhGyYrQdkTRgS5XWK9/+1QMApDHai3p6DA5TB0rfHI9VyqsEQyfvhg8uV6+Kr+Kr5JwK2jGHGrMFHD53Si9F7Q2j8Pa6oKawmn1DY7Q4AiPCM9VT4LAzyv5FX3x8aP4D+0VRjHjafuVzxeNY7PlkUdED46odkG9svXRR06KtOAJB4brTtSJYbkeEhAsinjhjQNITaqtz1fmFSE8PXLvt0zqMe2n4dlRfTVZcGHbqoLyarDgQrH8xHxKmofD9C3xRaOMfEVK9TiDNLariC3bDNczcYWIenUJ0mtY7rvmsQI6ZhW3Kj7PfobiRs8NyYaR6LwfQMVWVWoziVULudXoYzoF54dtd/eG5ud+fq3sd5mYXoRkzHB/GTJORGRvGw6lzjwn/wU7CYjxXQ1uBRbf9tBp0mV69iKTKsdRP+Y4lteRFPhG1BkZeC3it2nvovTaxFd8qWrt8je79GI4TSA0JgqArTytoKnU+cK+/UunZ7MAbS8cHzItuWVx+hTIzpitjBVpxO9zu4ogQvu3Fje0oq1jQaCfk7A9EWtX0l/JFQuLB4a5afNTs3el6HcCBWKbZnVaU3nVoR2r2sw2ywePnsNDNukP23mAh0V4YNjXE4HszFApWDYttLVVCheAB7B2a2ExtNPWiaGbpZyRWnHtYjXT5MvyT/q6+Xkdnr5evwj/riYZ31mB3vR6v/hR+pfWg8pttJLTERU9LSjjOIh0DavinkaWhiOCruKZXvVe8rIr1LhHipyXHrGpfI6BFnzEeZU+18kDxHQK7y3PbK6qc1tqiEPU6Dy+NV1eJCmaD+XVyw7cxFgLWIFJA2Kn+gqqkEC+osrTdOoYNK8cNmQfbzbRueNVmGPIaXwlhfAZBpSDiFzQo5mC/nLflGMws0sSXErMimeswE06ZyoYYnwij4jJPLGp9IIek1jG+R722XZFWeYhMNCc+lgsydgNwYEFPnoeG7qWCCkyy3Rowi9rvifBKBGaHHNvey5NL1ECoSsACc6fv4f1gsBJFT3wdpNPS6uMoxjcPug+NeK7Nukz5O+GzJvgRGP6AX+VIG0LBAdD8wAm+NbgdwvGdk9r5nRMHWJlJCDhpaYQHMggFwjf4TYpCZQA656++ooKEEuai9/UUPW3Q4U11wt7O3M0889M6FY+zF397++Y9MrJKblJ08sCsnOlpfe3SrRuR7gisvYLt2pn4YI6LSHRgLT+oY3zYJmD0uZJJU+2aG/l9G8qnQUoJJ4EOWYFJhydsGHH3XOmTO/mE3h9YAh841SN2398RH2ARaFDdGegspqrvHVNTn+sRr7IjsC1baRqHvuhyeEIDITPnzN4b38HAr8HITxEAr9Bo4fcwaLHhqqFPHCFaWSIcDCHmjumRg6fCFexdo/bKWACrPkOjlodcC+qTKAzEFmHnyf6IqIrv1kTquzUwQeuVMo5GB7SR8xM+THxAaIANSApkU8j+ASv+7Yc3f3OqJMx5ZhdiOcaMvpzDhj+5g6/fAL8Zd+yHZ09e9oEJ6a450n8lc/P/I77GN3Wv0/g6L0CvJvhhqGJ3fYOvymGtUharrv8eLQFarlz5qlvxzsEkRuAq01myO8byBVikxQT+KK6eNMUn3acTzkjn4AIlI/xiknL4tSLankNBRgr9CaGnvvn56kd2cX7xFfuImYn4KRpUXUz6HTUrYTfJSJH1PghgOLOmt2p7rKZXanqmpv/p9kFf6cB1WvS8yYZ8q16qTV2LJeHwPm05KrDKnPT8o/OwUBQ2qEMKmgP4jI4geG8a4CQGB7ePW/pW3DoCWd7VkeC0cqPI/RwQiaq4gy2JC2t1HojFbvQZG/gRuJq0YoOvR714+QpjWEYep7hLBYzuow0LzmG13Eus/+Rh+6BOGllVA2hZUOpWT9TbIrYTkOzVvVnQzKT6T561CebgjF2p9J88rgvYwdHBxqUbzHdc7Rvujty07lNkiRC++MEhWSFGz3wsLB3I3fGEV3IZqWiGYqwJ2j+eIDQ45uiEQJ1kdU5vpCz34sxacUxNzeTjwZnZ6sKaJH6UENQSeLco0jvHW1XaFhj455mvSrLlXVHYRBMPO2BWyGvAKsnoFg4tA0LGZ7TaXaLKjeibJrUGOZrOKurORDCQpodpbKXZ6cqB1MiDoUKJmDwL0mMYEoD95EAMkYrJdcsYaHYwaK3eYBBRp2i7oVWGXpuiuU1unfXBYKkdFbMF6VnbbQeSPR7U4qMMp2CpWCF36X022Lgf5Mfn8MJJfNy2Q8PkBt/gGd195iT7S9I97V6zM/pi0hFSBGzKJ392vfFoOvhKrd+Bau2hhiOipR/ADJzxBS6ftQvnnBx1ESfag6OM+uQ9AJlN+wvgqX3dRyjfrAlKE3Xn9hZriemiprMYxh9S0OnFrVaEr/e8r1J8tTZ/aHzteJLiiFRskjfzizG+0OK/cuADcL/AiNTc2zWbydeSH0l9J+pr6OVGdagf9agaC9x36QOnzQ1ILWxzHnwvBAASlKjezmWp0aKn5/W68/Zw1K5PN0raNc677lL+QeAOlrvi4P0ody+KbcWnA624W3zjVpErLEpQeB6+jCnn97g/mHsuGuMHRusGdkjb7shKMA5PbwBY+CZNmn+KGz61CzQ6zrufKA6gbOtYTH3u0acttUVFUEkqbni8NrLE9Id4purrbNZfBTE6duL9ODr1dHk2cDKuFBtFeo9psd/CWjocoSM1DocxH3YB/lfRd2cLHcH9ROPxe+JNKoo+a+prisXlDutyKnuoJTg+6Okq/ReMHF6ofeKN735LNywSaT1RJHKRogjfBBdFMv+KXgs3+h/b+26ttX0AAA=='),
    'evaluate_wham_feature_substitution.py': ('6063f5dd58480b0c0c7bd144fb56f10e4ee206a4e0e740ad8da8dac30a08dc11', 'H4sIANJPomoC/809XXPbRpLv/BUI9uFAB4QpynIcbri1XsfeZCvOuRzv5kHHQkHkUEIMAjAASqJ1vt9+/TEz6AFASkr2qi61ZYEzPT09PdM9/QXsn756uqurpxdp/lTl1165b66K/HTk+/7Lp3+bNKpuvGKzSVdpknk/vH0/85J87a3TukmzTK29N0nd/Cv94G1U0uwqVXtpXqdr5f36w8u30Wj04Uq1w8ukqmHI6ffvfvU2aaa8eleWWQqD1qpRqyYt8jrkSQw6/TNXuyrJRmmeNoAo/ZwgbEikXFbFLl9PmmrXXHm/vH33k1cVDfXXkfc+ufEqdQnUqsrMnG6TS5gyqdSIFgEAn3YpdjeFtyq25a5R/WUVudfAWtRtsmq8Otkqb1PBvzWtMcVlNyrHWZMs23vqOsl2CTCPBlVqtasq6CauwByV8m5S4POuYYqTulYNkPtjM6pUWVRNbRcxqctkpYDhyWVeAL0rYEleNIS3TEpV/UftvX33j3evn77712tvq5IayN3CXEAZ7OFotKmKrRfHmx2uI45h+TgBsC43bBqNTFt1SVtkfq/qa/N4ldRXWXphfvIfaIh2wELT+ltd5Oa53l2UVbFSdW1b9vaxSbeKCVsVcIp456PkYmWoewVcTC4yDVQmDU5uOt/BT+5o9mWaX5r2l/neLuW34kKQm++25R647OWlIGFrn4tqdeX8iPI82uzyFW8ojnzDM7778Scz3Y94jjQdOMa057lu/LTemjZ8Ho1evn/1w48fXr/68M/3r72F52/gkF2nTVwns2cx7DOe7fhqW83i65k/wrMSv/rPt29//IDAs4uzZ5tvvvn2m9NvT1bfPnvxzfMXz15cfDs9U+sX35xdnJw9Wz1LZt+ewZaP1moDNMW07ACPopojd8be5C/AgihfJ1WV7OcjD/5LN15ag9A2Sb5SDBxqJnxQeV1UY4bD/yoFhyj3CCgCmU1WV8E4WpU7+JcnG48EHEyV1DQV4x1r0uqrZHb2PEYVEODezmlLibq6qXi6dXqJqmdhTl7Eg/QEKD10LKKiVHngVxf+GHcJhqtk2xK8KSpvdbXLP4J8eikogSBLthfrZK4hI/hnHbzwnngn09kz/Wccehe+L5bd0hPtyjWIdUA4nbXq/it1y0+BWSyI6Ao1Q6aZW8/FFoTep7m3yYqkodXT01yipZYABvTQAGvh+CvsI6Dnz0KQpnK/eJNktYI1fBpbfu+226RKPw9RQPOu01VzDhwJeT7vv1GdLZmQTZbgNjxoUmAnbBP0T04c5txZVvqgOstM1f4cpwgQeVQDZeOwBQEllvuaLQyBLcFYwpRnUwARTEG40DubOkAgDQNA3545syW3ncmSWzvXF83B5Dat4yS/zFQMcrVNmiq9DdrGuSswyFLZwIwkyBpYyV1ZCprlMroG7VdUcV5UW4EwhC3ZLiYnofdRqRKfP1QoP4jnKsk2sUWmH5540+iMuust6E7bAUq1Dsbed96Jmjzn/lUCN6+hQm3LZh9n6UcV8IBxC3T+P4RraYFBTQRidtM/9p56bovAYVEAfd7EwHFrVH/awTUcIIJnL6IpDfuE92aVg+K1865gawL9WNSSBDjmLdOACTTn2HDPnMEE+dHiPY+iKPTmJ0wmmgOwFdV+AOZkzjDNTREjs2fRFEhtoewCIhAxe+jhWlcVQFvM0S4HQKU+k2AAmYM9Mx69qoq6PSWfFfzk/SG0DHMbeqA6PnfmABNubYkgNLyKKSwEd2DyeaBnhj37bgccvCl2fB7ooBGT227PTA+Z7Ad6aH4eAsYhCGOzb4/gXgWnsGWoWBbtbkbUAO3qOl25HdTiKBiL9GveK5745yJX/O8SmB5YmecNmgjmdfdxaLyZw6L5mo5WC4pAtGyCMMqXlQWqDWPSxc/XAbc+SG/oJfIIfXaBpfNltMpg1qDVuk8YJqJf5/PJbBl6zw0dYnahw0zrgyjZpFUN+rNWqwIM74VFqYk6hencplMtPzQQBryJUM+hAQ+6mJFJSbWIJZxlt+6deDwUpZ1azKZ1dCX08xSj9gonlap3h+T6Kq3WrZrBvQucRXY1Ce2EVoZg/XwMuuCE0SigmeH9pSrWqk5X8VpdVkrVMRqIvAVgIfMSS/BA0lV/LwAn2OWqcVtHBw8LXGHpNSr4FqH3V40jaqokr8uiVsQvq3IK0Ow4JAjM+AgdDrR9A1jJCSyFljRDVhgZgZ04iaaou0EtghUIllXZ7tYE+kIEEOx2+Fcl6xmwQ+v1BBU702HNFqQzVlUF12NymaCJGrdaoMs0ONbDfIsPXdOHWCj4tjggNc60wugBXfcMRU6fLpofr+Ih66FHXQ/P6dhh2z2HqKXJrNzh47pKNw2f1gFW8fHtdRzTAweZY+Y4yJee/nDHW1oOIngYQxzJRF7ACYWbVXv2GFb4pdmhTg/A2XtbrHdgOWjfA5gWxxhsiGMgJ9sQH1DFtz5BvQO7EnSvhRsLRZVtogvQDhcFSRW6mqBcFFgN8RZIzgLHs3DcQD/E8wdiCoKwZqs6RPc1JuJVvZjase2Eqyvw5lWGVoMzN/pksYlguORZbxP8HxgGHPhFgSGSY6NLHnS9KvLr2Tow04BYz16gtq3gV4zW+wI26CJNauN7dBH8vSp25c9o4p48p9EDIK9/+mfQb365TkrURy+vL98VRQZUBCwZPcg3oLga8AX7PT8le1Xx7DN09dDPOx0AA5YnlYQJtTs4wHJiooksxRe7zQZOg69lmhyYUFpwASF64PC6WdvRsIt2sD2b4NXeJNWajmbI8awHyS3Jrglodc+KRmrPS0B4z+dwsdP/Tmfzyels2a7BXtFrg0seqsDgGXejB2LcEx7XrhtsKtmCjNRKDPyzNUCwwPKtdaVWH8sC3Mi4DSIYe9Hwg39pVb8Dz/O8L/+h8H9f5vuldnxb/K3PBkQEnXlDsMzKOCtWpMoW/qrcwe7dqPTyqqnjIs+Mc8xOIKBJMdYJrAG0La4I1hv4stsfm/iMM+irhSfjSCI4k6S18t7vcoyuvcZr0xXkjf/6tgQkwPc7ieGr6gv4/RhE9e7kTNCOUZO7znq/+B1xIJWGRltfsY4jDIQGAi7S+4i6ELkuuHnu6+0V3T6YlDXq80Y4wKQd0/wy5rgGehIiwOCwcO7wTnj9qixWV9Dd3QBul/EBIOZSDUByu4RMVitVAnsHgG1XHx7Dbrzig+MEiBwPrE3XdOoGRopOJyRyldTqND46tA8zgGGrmgR6k8PjLYSNpUhvhg5DUwTamQt7uyrl/uYqgRuwqLTVRz8xUG5k/tGqwN74LD71vo4omghGpqqaYEqnLrDzaM1NYV2MRRL1dYT9JsL7swLHs/roeX/yyn0GlMwxSYIx7AVDTMDrxMzIpClADV2rjPV5rsctDAbh8GzLbEE+qG3SptxiGp0IvyZW2wuwFs5OZm1jHmd47dWLUwmISnmB10nbWOV5TK63/9MvH9762j2Sgvt/ogjpODuS+1Ht5xxmdMK30BxyM2ojqS5oF/xlBOK9rYX9BRoTMyQwEJ20qqkxWgwCC+yMtFLls7hNa3A4LkNvlyujGhdmR3qaih6tOhJLgfk0Jg/obXE9RjVzYqhlNCAEQ3YFR1mjXtzphy+S3MVd+9xTzOa+1etphU2Klk7SxHTbBx3JIlkxuRh26fkSlRHs9tncnHgagJEWGbiIPojNUxabp2Dg4qxPjenxFHNIIIB7XkANy6HIlkwxRdjKBj6mDewxC3ytHKqijAkPWtBIutkaQpfWZL7j/uBv2ltVmeajG7XxXxW7bE1HSss6bxbM6OGMaQPn9o6vRnvT7SjG2lkDt/MqkIwA/xnbRWuqInULS2XYgP+MO6oTmiJn4+yOrvC4UlNcFUUT4D9iL/FBWzdJvkb1TpZgexoRHreLULxJMVLf+Rkeg+13YsrVfyzGoUEulIziUJLHLIeUhF3bvKsUbFcEbvc6BQ/OTfOAr9ikeUcF1eQarVAPVA5+VD3VABY8dmKMnYsS1iCwjSv+7BEI+MusuAh48fGT6LfyEu5QOqnOsDGeX1yUe4Y75rYldtQecOToz0XzBo2+jjoSp32TArWJSJrTRsC1VlHOYs9ZOEM4cCqD01t7rRra+MhOvtY9sRrvQmXFjXeHG6m1lk1VGTYwOEqVjo+1h9pc79dwmxZxuqZkXsj5ePjppLf4us/AzzrHQVpB2VlyTOQvLKaoqkuQZ1AqoEVOxufT5ai7NzrmzpTAEXVQjcRJGzwAXU3T34iN/1bfJcR3y13LddA1ErXROaCkY13ZAN4YaB2F/uHxMzW22hp41vIIEJwvrWQZtuLBtywWeUC4rZ4/EwtjQsklFFx7CoeB576jvJ9GNJ5Pz9ZfiBopqYwDuUY54o50EcFRUpYqXwcM2l7+KoPxU++7hefM433nZSoPWi4dw9lCnTtIlnKauiNy90pWKxU/F8xJz+XFF+L3nTmM7H25e+3Kv7wVaAFaiFoPmy/HNC93WuWAUUQGRicLfXFR3Lot4LkX2Y7D0V1xOmoD6OsF/DhKgiFmCiLMlt3k8akOP1O9UAs5m58eBF0VBejbXF9bmI574gV2TSY5svQmmgDEx1mcOaUpcSrZposJ1qhOrshSBbScC24ZAGpgHPZbT/RxyIpclypg5thBxgBiOyxbmHT986nFMZE1E3azzwlpC2ZptS1Lkz9rGSYuSGMsoXFbRqCT0UpAJgZBj7aQt8Ni1hlWmwO5HUAidsXGbFHa7MYAklMsqzCEdJFukxo9oM5Gwj5+502jU3nMbwdORkjjbZqraPRSwGNcFcFvveM+WAbDYEDEb/IonXxDea3+pLrsoQ1KlCq7Trvjz08w93mypGwdFTHwsmc2V1ULFHqJmpKJRgnr+is7XWm+URXpAjSkweHB9YIxUSWrxoTG2vCd9rKodT4QnAnFrdq7ArizpxV0fkQYnvNHuQcaLTo2FDemIpBQxzX7vnoo1Y2bsKBt0LoGHGvQb4Z+CSeuMpUlZU3+3VTnpMiyQ/8Q9SwWaAkrOIGzj1EAPMWCRVgZZIkXQZG1qlcLv1szKEzWTCXXSofzxU3J8QGyFvY1OKtgP6GtkDTN3sSxxT3V0JruWadZGXvksIG4us9p6d5DYk3nzIO55sXXYoVLN0ZOCvpB0J3LlexEKpfjci3yz6hcq9hVqyHT9fKC1YxRhQyIWuca4zP++7//DU1iq/N2cI5ejN2IvtVsoRdTOlQ6ukNThr1GupFmy34HXwbYPaN7ZTaFU9UHm509f0CjS7feZWOL9IbzvpMPySV3gk1yzd37YDw+MKm1GmVeW1MxbqMGWHaaxxegwj9iOKJVWzqaU3H0BB1nLFbbgPbdoRYWgRmZe9ARYm2MOcYfTRch+d5i4fmr3Trx3TOi8/XQEdX7fHVVFTmWCkjzjKX960GCQLdqgmVJCOoQw3VDKTPP1DgOZLGxNEmPhfOop71HZVe7vBvR1DGaeRuY5CPSqRAJ7W051G6IHuqjJOXH8mAXBhYPdrLT1e9cwRpAU16rrNvbqSp0dNVc3iJOxE0HhqA5uDXFLXy5j21mHhZxCqd7W1DiGP1LuAEFDt0BPAeGV4GdJjQsEKgokMCP15TCsKGyKvmNva14rRiRPSnuzGHLoVDwQxhfWBh+WSVubNE0gn/QxefmzYg43BxNJxlX+hmmM0/oZq8G2NCjvqUmbDe+X6Ih8ikIgFWUlpC2C7fCdNG2tF3IENOFz2h0n8yXMnuhQJWkzd5AwW+ZdcF1mi5edNsJC1dVYnqRDaKPWWE7+adbzWnrDnSqIaAik2NlsZgWnB9gEHqLbLNhIQJM3JbbMt5xhy8HwJBF6Gm0wBfFen8EGPjZKVPdJmkedIoVqKIfXQ5T3R+9rC53+IbAO+oJpNmwxRx6xcUGi96A79Um2WVN/YPKyjcGWBwenipK1us40UMCfzLh1z4mp+vyBvPaeC1xAMe8eCHN6GEUzVWl1AQQTOhg/U4s+tKZtHH234sJ1fcEQ9t/CMEfp8NEB2qDIKV0Nu/T4mR6dDC/wDI48nQ6fRAnyfqboPU3iOb5s6NYwF+eoEqZYAFPwrlGfDa46PIVy4mm96Iz9Ws9vAdwTqOT+5E2KoGtqiZUQ3WEwLNjBLZiPZm0aV2gM0tX+wm4UCrzpUfBKH3UYgrkkVSkzcjGPEwMuAKZXPg/7LZJPsGXGtCzwEv2Gnx0jB/SBGD2FPiiExnk+AYRylV9VWRr46wc5QUbOr/3rIINZs/rAKrjRzVJs0mBK8N7GYYlK04y0oLiBuY2cdDqEu1LjYf+ICbMD2r9zfRSZNHNfSAYj1jHqKxCt8d42K3UdgDYpjvaO5ANw3bMfUn+m3QieHWYh2Z/yXh16NB1VqFDzpzCNkHTZSc52ZrQmjl0kwQ23mxfgyOcc88H7w447Ud4uQUaibaBtfm63aYU6bXvekW0/Jh3VwSw/MsUd82v1DVfB/jjh9cvv8cCj9XNeuGyCI4F2EJ0pHQ6GrOuZWCTenL+rxaeeFXqd9XD6NfyCN2dQNYWxogJe3lWkQ9YDKTgaGl0gcVwqKgtojDitTIuBbs81gXjnwF7Prhc4e9gteg1SAPKN1z1GIf2KNVu7EUKKfvf63DKHeP68mdgx418/xG6Whq/+BhR2dVXWoo5aogagyNa+C4d5/y7MjKWYabQPMSmmIMYIgq3DgmSJ2v6Tb94U+CuX7nTnepc9ywHanf6sNwjYW39iSjj6Y+zncvjhTyHRgqg5WApTwz7FHMhNKAJ+nhEWc4S89l3X8ZUduPWlLZYuBqxPbLHioHk2l2EfTr6VUKanHYqt/YpBvz9mqIvzrHVsUlRCQHnFF8rjda7bVkH3aMx7p5ct6SmUzzkKplWCMNh/e0cSid8ZFC7MaVD6LU8UT0H6mGO3+Ebdm6mLc3X6jZEQY1NDgh1vcp36O40KmCJhCOQwvkTMTYLvaCyJYnAiawETubycNJzfCA5blZgoiRE8FgmPe0aH6yFMRkm3wu3CU/Mp9aMEF/NvrJvi68Hk9E9pbzRt7axkDElOJ3LClxT8GOIHvUTe/y2IkeOEBIuQhcrh4gNBhFjA/szpdeaOZCZpTm9yO0uftoZjq9UULn3Qk4sYnmUbB0uRzarOTfYKHdpyBgv2X7QP/FcWQqXI5tB1pZu3IaSjsWacUQ373D/CK7tfQigJoZjUw8l5H5oMERjs4EaslMALOSSlxXzWwS1SB+QoLfNd74mF1QnxoI9U83Kv7+MXDHHDTCbdlCQHXk/p3EiBW2zZWQ36GCwGfKxnA2NwRD+EDi2D4DbfP/QGNM5MK7Nkg4NxNSIGYPaprWTjWGjRc2RFTehSJLS68dlHOiySznQb1dlkjd2VcfhKTx2ANZRvqQ32OX2/iI10dDKBXDYdrr47BBXsQ1XMdHq64ytTPobnAjMYOqfCN2FeSFZwdErxnHzSBz9NIdXVOKci3mXQ2qrozzM2zo4+fHEZgdD2C/mGMhdSWo6I+RF7nY5Jmub7XKBTL6yv8SuAvl6YdYo8nqu0rV+gEjydNIfXVHsHVpnrYeqKVxKb2NQoxxuj/kV8uE6EiuEXDYiS0UEsqFF0AwOZWPxAvR0LEtUW5uDSgN6uAyZj0SnkwCn/PpZp1Sgx1cLLRgrFZbuHyiJsCPlJxFC5jHXk1AlwLSDi+q7xRvtjz4JFotD8dEDYCnkdw77BLGJNPASsysnQy85Di1qaOoTfrOc3vFr5cK8P/lIHnSW/2BJsNRYtWjew+yQZHz938sVxnI+pXj/QW4ITmDgge7SO9d21p4OmBj6ydVKPn7Z4rbThoIDzfin06PPrD/vM1x3DbDPzSp3lvCik5OXL3wMzK1TTfb5HnCdYxJ7cnhAm5qzy9Mv5olbEHhuvzfQQ6MtuIdnu41a52AY6pt+3rebGV9074JDCi30njzhUzFsbfw7kujSwj235u3y4Un1h/HJ3K2P4lPXKvj/wydj9j+AT72jorWte3KMQovEAqkcocdBPdxl6P3DtQBBI87NHwroQGiCKGKM0fWjb+bL1YQCe5/gh+GTyxvGR4kZg4ffdO8MEiRJtaq/e/Am0rGyOt2CKqjSZh/0bdOubLofh+h6loMFPHe9ch7fwINeshZ0H0rn6uZCWQ0AaVtTBv7Mp424rf8lpTb+KVik94JTzzzeOQJH0Ei+D6Bxdv4IGt7JNn13LxIQq8eS6KyYTo0zC7U8Ag2n9ofRcG5/GNmXAc3E8VAR+JQn63xywgXJnYD9oSCOOYnd9rEzqqdR9ahuuztKB3PsFAPC2Q3ldMlxZbIbyumSwdAy5uiEc47FHTHUmBf5hL55xdFGGFvjlx3B0/xYezeqUvbTjWtfT3PQPcParyF+u1mMg8OGGdsxeu0czOfxaOi6GKSmZWvnehimogX/48re/RjgH1f2Lr7HKXtn6L9L4TtIjdattsRerS65lvJT1QRvIuiIM/y20P1TjMfuJqDGaJEOKTiHxw74oK7V+UarWIW9QIMnztTON4Vid5j89ZTefZAjQ/zi3AuxYy5tjnIloEv9UoesrLqgwIKKu7cBKFc5+3cLjtIADfHQxSHrr8xaBlAOrdOOlHMcROKkHg9dMkh8yxCJ18DaS0SmyEwaDSu5siwgdkX8Vca2MpVeT3V4WAPGbRJfg6LjVZ4MfLEg3tA1jaZw3H5pFq0NDSDG0MTQRX9l/rQtJZl3/VSR++TClZjrXea89gPdPcd1eHcNlvt336I5dgoes8sW4b0mxX17TPssj8+qoPrHDiPxewq1wiPkdz9rTF9q1rXcX7ffXKbSBp1gw5cTOEjid9bA92DM9yBgxwi3vFe77rQ1RxFQK+kuSJaWsb5MmbOd1x94ndsyi/WnZw6AbBW+cR/DVa96aWkCeN/5VHJxk/PXXcVXk//c+2iy30f07uWEP6XcflC5XUDUeflueN/aiqv+5okaEugUv8IhuPbjCvzlWywqEN/NHcpYd7egX2hxBFcf+BA6UwMwUObQPVZtociRmWU5yTBX9e3YZ6ktlGjrY/VH5DqU4OWMJrm4q4enMvJJrL2uY/68eEyfF8c7ols67NzJ44EL4KGInNt6CNE2zXd13GYZO6xwHa6OgjysvVwt1xt/4FoMB+Y2l1tvZdRxYGcvy12MgUsTP+mvq2MzxTovxLR1sitDgtS1sIzo9QNbQ6O7FltvtA33HNAHQonyt4jtT+dyRzngeA2KAyCMth+xAoN/8FuCoafATm7i4qModpEjb8COVTGWzknXka2CkPLOebOYjbGu77+wIJZeksCXefxds5m8MCVkgsZYf8qD5VR26GgdWPQOBWlzBfsOl89t4Eer+tqX3+vuItYf78aq8FzdZCC2C3+IrKEve9Na0Z2ASaLv01XzKzUEDAfecaqyNdWQLDDD7zrQ06WIWzMmZh5+ucwJTsrOqrhxHXHjVGxLfvni7o9ZVo+4hO+5gA9Egvgk2HShvzw3ChSeCGY5oAl7kRzHzB/dH/mR/kU4+gMK6w8rq/tDP61lPjDqcaWHLszjygplVZwQZX3UhCy7VZ/vaYPn+B03K5NfOpWh70QZtse2FRaD9oTzi+8WUmEJNpbwcQk2f8kETCpDeTfy8ssejNPt69u0W5n4/vU/Xr/68Pp7+06y3mv9/5UhrDcyXc1tImw5Weo1GqX4rU4U9DimCH4c41swcayj+PxKzOh/AYgUi6TmZAAA'),
    'evaluate_mobile_pipeline_3dpw.py': ('9305461efdbe5c1fa0893b25536497643d636f5590695f3f8f56dc0820379ace', 'H4sIANJPomoC/7VbbXPbOJL+rl+B5X04ckPRb4k3pxlNnXfj7OQqTlwZ10xd6VQsmoQkjimSQ5CylJz/+3XjhQRIUE4ye6mUbYJAo9Ho1wfgv/3lpGHVyX2an9B8R8pDvSnyi4njONe7KGuimpJ6Q0lUltMsfaBkW9ynGSW//Xx1Q1hzz2hNipxUdJ2ymlY0IRdvbn8j6TZaUxZMJneblJG42JYZ3dK8ZoRKquHjJtqGKxrVTUVDpFSndVOnRR6Uh4DcbaJa9S0qkrIig0Fsgry8jVj9a3oHk5ZZFHO65DGtN6RYrdI4jTLyQA9lkeJ0UZ6QNE9raE0/R0j+B1gP8FTkdNIwCj3If398/5GUBaNkVRVAi8KYMmsYXziLtpTQfZkB5To7kJw2dQVTmERJJFhLP/6CogrIuxrZKypgoSpq3mfKSuCWJGm0zgtYbSy4SwpgIi9qEmdRusU5J2VU0urfGfnl5vY9ubn9r9vrk9tfrwnI4bGosmS6roomT0DWW1pXQCfA7ZpMgPstCcNVw0UawiYgAzBJLjlgk4lqq9ZlVDGqnmO2U3/+zopc/c0OTP1Zp1sqZiijepOl94r8LTyKF/WhTPO1ar/KD+10vxf3MEI95c22PIDESF621IsqllRu371XJN6hFknafyRb1Yx/i9Ymw704cFnKl7iZ7bzPKxuywV9i18lk8u7m6p/XH67vwpvrqw9kLhgLapqzonIXp8HL1698Ar9eXfJfp5dLL6go28COuRc+OYP/Xkfkl7s3Fhrn5/+Bg8/PX4pfr2w0JpOEroQdhXURytH8eSYEE/CfHpn+JGe4431mEwL/oqqKDjB3XgYR4w9irE8S2CY6h/ZVVkT1xbkXgAxzhvrvngNDODs5IeevXgWnnFRFQWo5ccUkKPeQ76DLycL8xBAaDtYFIFfC/mgikH1cFaXLyQ6X4vP2GMyZVuF+RjiDRuPBaGRpQnsNn6EB7B4Wfg47NOHC0WYQsmFxBB5szscDs5yAi2M9fb2cPyGbVVFtBdP4j3f1+WSe37aKWe5U9+Dq7dt3H6679934lgXfaDoNTs0GJQgQsOT0vN9lMMZCV4luhIy2BNDBCB31XKzlk3gEkw7+/u49LObqU9d3lWZZXGRFNXdPudKcSkJKc7dFQrMwobs0piG4mwbdtCueZ1JhxRPfJNy0/yWsluor9+CUpCsiegWotmQ+J07cJJFDaAb+2onLBhzf5D8FvTRfQQjKYUKc3fUkIxiwwgJCVbUTblDsBWo875jNuNeQOlQ3CXA667xCIOPNzzefzn8Rb/1Og0N0h2xGMoh/C/SFS7+jfh/V8SZs1dKYwf7SJiCpx3UDe7PQTd0nx56SNK4XIFEfXfFyKQS7D6viUbELTiBPuBUvwRwWS95jG7GHZzuhGasO+qywhx8gsuo9i3wFagebovpze9N6cEkxCh0TBq2g0xPeDmYEsorAjac59/mdCYGVrSkqXkZzV9sGz+/LXVPvhLJ47vBgryuD0/XIaLSj87cRqJY2LGXRPRgFhmiIhwGrE1pVAbTW9cFVWj9r+xdVuk5zoCGXq7kfbdGCoZrGIL1QZEvP9keB4DJRHtqiF0JIMymsF30RLGeGP+CZkpikKEF8SMPDMMiKpoqp2VlfEHpM3iWArdrRqnadT//8u+ONDmABpEKQTLmqAQZC2PCGI3qSUOMG/XTqGDJhda57+RIUAX7A5o86ruGU3qRt4nKDbGrOk5wAcq9VGEOGBW7T9XTvCOkGamjnOIISMl4wM5PT3mpMd5xu1+zzHHk2vTSYyRxU//TM79FCDzA/5k09c8QmylZzq880+8EO3sNC+ureLXjM9ZoaIqwfXwTskMcbyKBxVzoyhnm/sMoYQ5PYg4mh7GqrfSl8VPzPadnqE1MvmNfX8aTe+GRD0/UGk4FWZ1BjzO1YSRLBfbGHRBwqA3RgmGujb9HfeSiC06F9dB5Oei/9JYapY0NCzFApapU+FVrYCrx/HcUb1xOZGvyGaAc/RfI11Oh7yuowhbpgD9QgoLiY+1XrbbR3B/NZbNBYhphxMGzRzbHUDEgKUqPwI0riFUpRLquryKSEh0IRoUnZPjD/mVYFcy/+ZslZh+y3YUsjANMw9+zrxvOIpsYig8ck1Hbsmqzd6zRvqCknJYgwieoIJD10cn2JBdhTl/y3KIZnnRz1zWBkMfPJ7HxJ/qoXDAPOFtyuIIME9wU5pLIv+bwcyvkrWAkNvRswdb40nRZU3IneTxv9E+rchdEdjCncH/aHY3IWBoe9vlfG+O+44J4Xnk+eka2VZl/eviXMjT+ByaKb4DINWLOFyIxC/NvQNLdpnm6brSZ4tuDDlgG8cqN9yuanNpvcjw4DrzQ2TBU+fle7wP65iokXiq4nahkoYAvMnCy7wgueOfZ3xTaCqPBBMTZVK4OV/5WcBVD/ngU9jkY8uCrOlK90la4tTpfAYvt0vlRsjtE4WGicGTQuxmloK7RqiCCMrzWOYN0at74+kf7qbGnxk0LZQUqv7Cp51i9Je8mW8IxFlUA8rnngg3UBxSH7nbOa6sa1GKqHxfP09B7Eh5IyGrMCighWt9JHIRmJQ2/1WRELrG9+3ElaxSIW2WrNSTv7tDV82eI/P/5gjG89xVECsvYf7WPxMMe9iykcM3gPSAER8NQgP4q77rqaBnTQ10vI4JWQPQ8kjPO7o3s6ORr+XVuI+BEjhI2ySczIBfoQnA5kdQnqUCl9LnKvT/p4DiEKX4FTshY6FIkQ5qOcMxDT2SmCh2KDJC6ny4g7WYxkkL0jmYVICzGn5n/5fImYTlMIaLTCPRGkeR7H3wlQmlfzOle9Sn1QqBulucEHsD2EPrTEvWPXGDYob4dEuhyBt7VygxHxAzgMXJkQwbKTAa+i5ZRy0W0nffHegDj/DUFH1l8+9M3De9DcByg553dVoyWDX1dbFk1dNuiJ5NpcPsW/vBTrb+Gz1VhvIJcPCgyBCcGzmR5NepBPaIGhNQsYc9sKcZ6YK9OgZxghNlf4HU+rgcf7tj4CnMt9UWSuNkqtsGv5YsjWWVXRljJnNoSczH6i+OfAEof1XHcgjp+wNvJkxjU2XHNbjg5GwqhtVOG+9un2SfHCm59ogZdRew609IK8N2QFnnGX1lrnnsp0/Z9MuFcei4X8GGEU5R2eU+AC6rQ+tDZLD/wQxOLafAWGSIJCVeTRFozvZLSN6irdo89Wr8NLcORyqu6w5Uwctvjk0tA7NSig+zICBy36oMO99AJe1a2bomFue0gjCcNsRf0ti5fzPcd4T7W/UUL95V56zwDm7aHZDvQsyiW4lYP6ow+uxL4/8qSDzfjxn2jKonuKqKeJO0vMm2ag1jSRGCfYxVLH0FFwOqlvAOFzWj8W1YOSd54HN0XSqEOQ70ThubGHWbpN62/B5nsrF2vtwfASje0wRNBcxKRd6OJKqcqcIcqykFZVUdmQdh1Ah3SD0T8a3MiZddKubw16lYXCmWEI1xo7z9V7YQHoNVKjaQHfuxFcv43ArWK0Sr4DcypCXuWjSIRWLRxodpYLBT21bptvVMon6HQF7Ee4fjVYdbNQiHZRmiHAj1UU1LJmyg++vp0CA2MPncX3ag6UUkff7GtEck25MBSYMbzPjtbZ714a9LohPw7wyRaDatFYjFmGsJTmhFpQc3uQtbJRv90dvxP9Yqax8IKcLS3LhowTo6/fxVn9DCaE+MwxqdGzOgNM5mbTO/fUXYKxVn84vjN5K5HR98riLbsqLoOEOr4ms7Godzjg6in9qzN7Eu+TPRTmHp7hbecdoNx35a9fdyxAxramdTuvlv+YyPi4kZgKvDgj/X1VdZOtujsO+HWMtxRFQL345mOYNlXuVlI1ubjgERdVjzsZHebyt7mf+/l+cTZbBk0ONR2lkFJxLMl+rILqO8cfXz9E6fq8TZ+/eigqVPhQzgeKdXwEbuHckorZ+qI9z22pS69zDIKFmm5Hs7muumeaO4Kkop96/D8eKBlx5atLGBFIDb3hzoC3h9E6SnNWhwhJ4mqznh4JrVM2YhY9vrQ9y4q7CK6KffGk1YfK+25plLdQlByDbfpxqR7orVDLl4FpOjKJg1QeM7ghyOMogtCjde7DXm350236sJNR+wz8+0J/b4GbHG0/EroGEoZwzAFPFmkbuQ0ohoUDuYxlb4yW+tjH6Zz3xvbPNy2j7ZVYn5ClOretoFeiLYUDBdtCxEJLGbXLEimj5BOYBhjKNb50V86HgkhBqXQ/4YnZF9SSJ3mo39pMF8u6CcyzEJm5huDlz19dmgkGbwpXENnbBFfm05kW0Cdf5yI090C3JXiuOIo3yj0MgIMvlpKYz4Y1teAlGNqF1k8uqOsuG/oDaMVgj8Dvxg8KJNCttV+bi4sUUlfDNI+zJknzddjeKqWVMzMU+hjM0FfiUUwBgT5Lf3LCDxCM6ciZFU9o61KuBAYy0b4SEMVB+boemXXZhBiLNZDhy4gvUCbTsjwOXejwRSe5MfCiHYHcq0VYez0NEI++I+5de4M44vKSsDvk5jdtK7AJdes2uJI3OG75G1e/9LCNaoSP4yxibD4Y8IauIrzv8DPNyreq86Rzg2KqIEqS7paIM53y5mR6kZSPjk94wsmLbTCYP5q0oomGWo6QqDcVpVMgMMVU4XupyBR7CiYbP/CM5nsp4ZZN8Yr1nyLw5/k4FFmxez2V3uHPUDm//LNUlEq2BBC6AIfKdUadtI6MlZHx2weiTU55yTTFkslKQQG7z2jFM1QuXx6lInK075UdmHQrPwspichUa34BTJDgv5AIU8i3ms64b4AdRN8kRAv0zTeq6uxUsddBlDZjb4X+hVJzLC/PL0decrpoQeD+nSy9P+EBj51ge1AeHF/zK9uUMbzbPyeIKsk7g/p9xHblMhHB9gByaR71vaUK7pLOTMMjuCh5qHCdGzlNSw2HsxlxoP6E3XCC30ECriTiSaHL0m+LeMqchKG4+x+GrsOa+7IqwB6YA7k6SjAUO9ttzsJZp7jRTkV3wk3iw8/XV28gRSXxYzI3ZQUqQfc1VyeJroI80lLmH7BCnZu/6GkQfjET/uPjzc27u6O5mVlCOtf7kgNk4osbSfiLleyTD1vS5Il8Lfo+Ob1MWeRXbVInHl15qxoWoCVYWAupdB+CWnfl2utht0a+x49wFZzE60ouQh5AQtB/3oaAQJHtqCpvBAgBdMSnIkAkSty+5Xg6ctudJkLWBXkT106diwhSbXmc9qzFSaloGi8RgyFNC9Jgqki3NN9uwMPJtuj4pH3p56F0b9yHlGe3VfQYtiipcYZrgKVG4e1qm3XCgVWdihdUrMxScIShg5kfgk+4+0mqoJelXmUodo/qMZYYYu/EJ2EiNxZjoQ2/lgIWhBIRmIhfmT04rSGJrVIRrYdtKtiYX30RzEyGl3U4HiCBVJOcOJ5WQz3jtmqqMPC8DLI0519tqQPtdgTHd8UE2ikm7K4KUj0uF2oknn24ahZPnEerR9zMlgFZ2Mn6fXCuAjS/PMkjsPau/ijsj7PwOkeVMTiV9kmLI8JIzjNsx7eFFi2PF93PL4fdu2AzvBhfVrj0lfquENVdVJv4AeEXlK6Sl/ekq0wQBDDFKmvYpne4Lm7w+UpG/GwX1mw/wGohwUG5NwiPnUPqAcSSP38MHn8ek7YCkcLPHYOmDb91tJOtWhwHrpV2LVAqy/YWcidhvFEDAQ+RJl3KXm9XjenwC8IgabYlG0LFX6yXk55Fqex1rGB20X8xcllyiC+p8bbKdrlwEOv4naOuvP/SfrOLk0aAyrG+tjDzNGxCz57X8/4Vqx6e3JqAvpXCS7wWbPPLM3JLTZtefscytWUJN7ID+7bP0zqDf8k84tNZdHFdHwZhYxuFO1oxIAu7p52vwcuipAMkwcFbxIyiWjn9L5RrvDwnPtCA5LL9cphnWTJsCRfg9KEb6QRMsKd1Xb7l2og4vwsRNeCDEEYds1XH/LBYsA6MrdKK1aJKEx8sn7854Re1XhAE5GFRRAgXGtQ3yvz7Yf5h84vuooUo38054wgTCJyL04rydZNFFdlRyOVgzA8Q9cmb218/9seJ749DyAlQ+M6V+kgc54DiPIMl7Gh3R6P79PkHnkfcXk3F183HvnHGHQkcKxYl0o12ei6W28MdJrBtnIsqShom0U2pDySK4wY2AhZWbg4sjaNsmt5u8LuPf0BiR27eE/zaPI8PeBMMlJM1qDeMgmJAe3bQ+dGs2YFqY0fzSKDpPV3UcnKFOIkn39avzRY1ANKKqNoyzAHaOEh4n6M6cFAjifMxd+X083NE9HtN/W3tcv2vWrdeG9h3RHkpPN2Qf2pvZcoC+WHDQpnuDKJE5/SmrZ8VNJ5aUCCQt+GAIVhesH3AzFk8MOGxCd2D/wmLBx2KwHvrG54fajFTOD+/jQkeFr//I/2jPtljlUKSg/WoqyjBNHlcIJQ8d5p6NX0t8+mY7fjJvgAhWaADmLImRtMzqKc1JBkNOMa96wRAQJLi3w8qeuITQgdRxZw+QrZM545jYYJ/X1iDI9t2+SBnHyMJEAveQOb6G29wRT/I+FKaJZgOsDm/3o95BxYmXo+CkMOGRolxUKy/xKFul7f0cpaxfMUCS4/qiKEXgyGgQeNqNT7me/XSoCXUidte5Wob7NlO4DS1kCPUTnujuLg9f/FaWBxquTDEfQxDfqAThgiSh6E81BGI+eT/APzdfQOIRAAA'),
    'export_fastvit_normalized.py': ('9f1c2a017f51b35d430f713e6e06e0a59c7c804607338f4b98de21219e9e9c92', 'H4sIANJPomoC/9Uaa3PbNvK7fgXKTufInsTKcpxJlaozbh5NbpzYEzvt3Hg8HIiEJMZ8FQDtqKn/++0CIAk+JCudfrlOW4vE7mKx713w229+KAX/YRlnP7DsjhRbucmz45HjOK8+FzmXRG4YiWIh4yRhEXlNhfwtviL3sdyQWAoiOY2zOFuTLOcpTeI/qYzzjMTZZM1psfGB0Gi04nlKgmBVypKzICBxqkjTLMulghejUfWOrwvKBaueP4k8q34XCZUr2KZ6FpsS2NLUCyo3SbysSF/Ao16Q2wLZM+9Ps229VZhzliYyzxNBqCChrBayMi22+CorqlcyTuttZc5DQ/zi7VlF+W1K18xsiQDV+ywbjUYXH85ffnxxFfx2evb25enV2/P3wdWbD68u35yfvbwkC/JlROAfJ6WfgyIXLIgYSC9SosHfzpwc+Ucn4waKM5BFfMd64AA69Y+mFqhkNNwwHkQ8XklD7cQHiAdgLEyoEOSMrWm4Nbq9lGXEMulmmf8uj8qEeXNFLGIr0CEoWwaBK1iy8sjkZ/I+z5hex39EWTDuen4N5zVLgOEvaXi7BAw4MkrUDzmjkgVpHrHErUEV6yvg5g5oCDp74oxJwZkyNRYtXtNEsDFqKVDsM7GY1ridDQuef4LN4CxngEu522LDRxorYAHMUozJ0XT2xBvVhwVTu6c8UihjUCfo12uOCvyUPGt2aVN2NbhXi/gSDDSmiZHxm3cfZrvkDB5zCvYTSxYiX4SlsZTgestt5YdBJZtNymc+mKqLXkEluWNcoPfNPO13/49qCzcQFhi45ILs1lWbPaFlG2wYjbSyL9kfJcgWXrbZg6UXeXY3i9xqG9D67NmY3DIOT4GI/2SLozFZxlRohr1xl8CvPC+L9yBw9+ipwh4AeXX20e2/Po1ogU57ere+gKgDXLhPxuTY60O+Bu+WLBugcUa3jOvdZ8/I9+QJ/Hc8AKat3YIx5j3e5SmcrcG6IE4sy9UKrMGREIqZDFJGM1Ckimn+n4znwlWEDkQXMqqxQYs1cm2bVd5gUSDzWzjzsLsZzQcpLXqWYRy1tg7tfNfzMdH/Hs/mk+PZjTfou7bxuNYuBweC5gAVY70jaZTu9hbi9xqxkRn5d+sNKgFCCTKT5DQCCB07FElMfXOV8bRyaZLk90GiYvqcLMHSgDHtfXo9DFkhAwhaURnK4A540LmjC6wihSyLhF3XQWpMhORjiEShvFa/IKPe3GhpQJoJb4s8zuQcX2O0UHpHll3kckxAsEGSh2q7hRMWJRjHPYvXGymCPEu2xukUuXhFYhFnQtIsZG5DXO/uQfkQEcdIAiSCEQkXHCg+bFZqsVM7qi4sEB9E7Dr2stMoC7hoIX6zaAe5oDYgCMbB3cyZt3wRgp9g5EMJwShlrzjPubtyPmYQcbE6ANU3tZW9zZx8sR+/4Q+ObT9/lDEH61rDmUVdPtQxmNP7IMxBdMxpBwYnBFExxBxetgyXp6K3rGoNbkq27uISXEcoQ1U1ibX8UP+q+O2KXr0HS/jy4LVhgwJz56JzYj8WolwKwFSP2hLA6tuxXq1dZzRlN2BH5IqXDD2Z4Bs0kTbNgbDYOMYAy81ih2/tXWiywZKKGLl3IAtGDJIe7lYVzHpbp8bLebyOM7AjTUBFk56JmiXHq0+ER29EZVst1NZ9om3j3HvCYkMFOw52HlSFHhNDhg1R2QVdijwp5WCl6vbA21wZRoYq4jFZQViB1ThbOZ43SOinBdlbeF/vLrhvegQ7OVY72p4q/LCz7abwzx1w9x6HnbIKsVUjoclgN3HwMXeT+IeV2W92Hj3jQ+vJOM7OHEkwgiAAhJuW9fsAU0Lt4Xnz3o4DSWDwjM4HtioFtqsyJ+yzaZJJmdUx4W5m+ym5gua88nHiDNO0eEdGIeFE6gQYL3zyERibTDT9iTnQxELZQRRztYqmlKg4MZEbzpjFG5EbaEgEEBGrGGIDjhHgREkcxhJQd5AFu5CTTR6qdpegqQLRH6By/Q7+X78wbJIiB3Jbv0+rbUGDERk2CmCjAQ0Hs+ns6WT64+RoalooVeRqCwbc3Z2c14X2TbFWlSZWEXM9VLncqOoKfi0wtts9UYMWaLsImlM5N/0SoM4Uc5UmurkaxQBLXcl0wOrsYQnHItxLLh30iBVJvk3xjBaWVYgOmCgANA/d8sOoCk1NbPIkEoPxZ2+U6EFjedeRgg8a4VLgiMutDcXpRyXoHhlkxNGO2NIsFBw0B2XfGVgD+HCrsnt4DoVImHNcsCvxL81Du/TTnYu2nXGrZhxbdjIa2cHMbghGhwYlKxiB72pssk7yJU0mBfYJlrMvWUhLoBZLsqIxFrPoFx0vd0x3ZWeHKv4ghxOzh4otEAA50xrHkEXXGSgiDoXl7VoqjWMOTrFaQHv8ccDzurJ2NH9Bq/rXAglQIMHdkSqR6pnPL1vJ1FjwfV1XD057lP2RFzmI5t3ZvwSZ+r4/OzkhH379hUiWCQyzUOY18qiHrvAjEzj8EXsGPuPqCHPS7P71Q6BG0ObX/gFAW/WtKYI+k3s99Z88OxnDcZ+cPFV/pk9vPB+dmxbMPRqroQX86x06sxh16pZoYMvZ7EfcazZ7ov+cPLrl35kD6JYfkhbo0Z+SieYXpeDBSyPPaHggUc0Fa3qeaf1T0Lrb0ZwalnPYsRqc+6d8XWLQvVArRosazKdRFFCz7jqTiWm9QUxVN6Tsf4xTc7bAkcJe9LyURSn/Lrbt9ECDhnooIEBjDNqksurCh9BryTl7iherBx0g3ixuWFIsOsZzisx1q5a6PGlXIHMoTu2CpWp5uvHPKr3HiDL97jtSFeektYju/tOiJmiqWqKqWr9L9moDNU2UQ5GFwT6l/JY0qZdUqfe5OgxE+7wEWhHQFBK7SDV9qBVoh1ctoMN0sCqTZMKwBFYnmECsgiLFknFtEvvFfl4gOhSzSJAcv7z4HWNcxPLVivzn8vy9T95KQcI8kzxvXUiBC5eJOlBHOixdsgizKxVWHkEN3rEMk/5zzFoZuwP5ouzEkPB2iQVkgeWkkY76g/KBLmBk553deRqQ+wO9irJv3NMyYyuTLxSI/WbcGUIMFLcGadeyfbZevPfDooT8gFo2UUVdAADAUKIzSF4bS6GxzzQtElbPByGJRVghTccQLiETmDisfsP/vDGJlPloaNUqHs80B2AcoQq4eu1TLH31ylWsjautjMgDtWaleDAk0LsZtsAB9WOjBU2+kWucQbwTi+uWiQGiOvwV8NgvR3HctHBUOggUutNvsVX2WRheffXUBwrzJOdBQrcQdReKWXjWjz5UCm2EJn/dNAs6XAP7gH2lEqJiWXNYDc8DHNA7noWWQj2Qlmlg1fN6LI1c6F9+fH559LRBCfO0UDMfqG1jvJJC0PrBf312fnpVwXc04dNSbnLMaM7vb07fTYCy04GA4p9jcy9CHhdmetUIfuVg7TlRk/faiN120e2ZC2xzS92+vHZsB697g4EJ2UAr1ozIWrO1BmJgevjYaNFE4F73M0B3oEVqOMKQGjQxet4Z4mMT+TCqR94QJjoIgQ7qGGMx0bSrxw4sEMObe1Vwi7ZT7CENhRjEQsk+S5dlYQ5BYb1wSrmaPHO6NaAioPJhW/ltuloqTbqoavfOKPOxIdUgOETe4BNqTZgZVh9EVb3eqMM1Zl7VhJRqFAGyDPA6IImhP67kbx1vt7jbhDqZFMc7Tne0pcdsDfF2U/vPjNhUW+zoNjBYl1AwQ7eSdNu2xpuhbwRibIWXxEHKJBZB1C8L+GOF0858435D0yppBq2bm3k71w6g6YuawZEGOEM/gnedrIYfHzCgb0vYA4e4x6J8J1/dYYRhqsuDXqt7UsfbTfGRKc5BR95LY0ysS7vDTzo8HjqInSHUv8lFc1emb7o1B70xfQcKBL9P5J1rNpvy6LFR+SCu2m+0Z4htbd66pQsYDnbMVz8H7b8L/atZ6H/DdPj+/duewzffdylzEAf77mQOZ+ORK48DeXn04uQrlGJqgz0fmQ2y1K8pTHLbRedrmFKp6Dgq7k0eAya6qe0A3KFP3AbP4jj95IqJVSet5v1Xc2/63aF5tOMMDp33F1jI1PDAeV/5hJX8/nG06cB0J6CnRsLXTzgFy5M75tZfX5j37HMsJLS0c/t22azFUPzE3O3ceekPM32eSs6Yq0Ebi8BjtOENsTJL4uzW7G/eQU+NrWd6i9voB2GmTIqvIL8d6ud8QeEkaANmd6+eiFdfkfpiKyRLXU990vGS8vs4sz7e0EP7tuYvgWik5ihmWAudf3gLnZ1fzdFxFBDLrRqs3MZFgd/tmdl4Zy7x7uwdtqjYFWEdjhhq8E3voFqiS2iPoYpOaXh++ZzwMoNdAUI3HIzDWtijmGeKtXc0hD1XyKD6eCZJqhm+YRYJxxcb/HLKGZis6lmkFte35ArHRAq5XCZxqKeeQBiYWEFvPFa16afSAJl23Ayt9RfAo6ohVPcCqs9WzbnwjQhsNY3r1rHMYtA0AL/QLz7Cs//i4mNw/v7sv2ZAv14CPdM2X09v/ILxFEBxfjAbk6nnL7fw5Pnqg15jV6rx09OCLA8wctnWCwKGwlHxqb9uNNTbNCjUlhQHH1nhU0E5p1u31fviUSvNul9a3f9cfyns45fCGhOO4T14150O/MZqGzDUqkwMW+oSHjaGl/iHLoVbcz0xrHmeZ18AARhew4IEmWsAcHAEpp9zi/jP+Onwyd4bopXTWL4ydFUBV5c+LtAiitbiS032wavmuOYaTHvRF63xh85im/wcuSN/VRdHoBBJ/8JGtaY+95+uujQuU+j9qy8GydlMtfeIBXJIsIxdq0/0KlHM/WNFYjSK8eYEg2gQkAUEhSDAkXsQmLCg5++j/wHjWAi/My8AAA=='),
}
for name, (expected_sha256, payload) in embedded.items():
    contents = gzip.decompress(base64.b64decode(payload))
    actual_sha256 = hashlib.sha256(contents).hexdigest()
    if actual_sha256 != expected_sha256:
        raise RuntimeError(f'Embedded script checksum mismatch for {name}')
    (SCRATCH_DIR / name).write_bytes(contents)
FULL_EVALUATOR = SCRATCH_DIR / 'evaluate_full_pipeline_tradeoff.py'
EXPORTER = SCRATCH_DIR / 'export_fastvit_normalized.py'
print({name: digest for name, (digest, _) in embedded.items()})


In [ ]:
# Fetch only the pinned WHAM source and weights needed by this evaluation.
import subprocess, urllib.request

WHAM_REPO = SCRATCH_DIR / 'WHAM'
if not (WHAM_REPO / 'lib/models/wham.py').is_file():
    subprocess.run([
        'git', 'clone', '--filter=blob:none', '--no-checkout',
        'https://github.com/yohanshin/WHAM.git', str(WHAM_REPO),
    ], check=True)
    subprocess.run(['git', 'checkout', '--detach', '2b54f7797391c94876848b905ed875b154c4a295'], cwd=WHAM_REPO, check=True)
actual_commit = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=WHAM_REPO, text=True
).strip()
if actual_commit != '2b54f7797391c94876848b905ed875b154c4a295':
    raise RuntimeError(f'Wrong WHAM commit: {actual_commit}')

downloads = {
    'wham_vit_bedlam_w_3dpw.pth.tar': (
        'https://huggingface.co/camenduru/WHAM/resolve/main/'
        'wham_vit_bedlam_w_3dpw.pth.tar?download=true',
        '2ba0cb6a7dd597023a6b2ad6056e7a8b6b33144a35fabea570bfd00842cd4eaf',
    ),
    'yolov8n-pose.pt': (
        'https://github.com/ultralytics/assets/releases/download/v8.4.0/yolov8n-pose.pt',
        'c6fa93dd1ee4a2c18c900a45c1d864a1c6f7aba75d84f91648a30b7fb641d212',
    ),
}
downloaded = {}
for name, (url, expected_sha256) in downloads.items():
    path = SCRATCH_DIR / name
    if not path.is_file():
        print(f'Downloading {name}...', flush=True)
        urllib.request.urlretrieve(url, path)
    actual_sha256 = sha256_file(path)
    if actual_sha256 != expected_sha256:
        raise RuntimeError(f'{name} SHA-256 mismatch: {actual_sha256}')
    downloaded[name] = path
WHAM_CHECKPOINT = downloaded['wham_vit_bedlam_w_3dpw.pth.tar']
YOLOV8_WEIGHTS = downloaded['yolov8n-pose.pt']
print(f'WHAM source: {actual_commit}')
print({name: sha256_file(path) for name, path in downloaded.items()})


In [ ]:
# Full 3DPW comparison. Based on the earlier two-detector run, budget about 45-70 minutes.
import os, sys

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT = OUTPUT_DIR / 'full_pipeline_tradeoff_3dpw.json'
PER_SEQUENCE = OUTPUT_DIR / 'full_pipeline_tradeoff_3dpw.csv'
command = [
    sys.executable, '-u', str(FULL_EVALUATOR),
    '--parsed-3dpw', str(PARSED_3DPW),
    '--three-dpw-root', str(THREEDPW_ROOT),
    '--student-checkpoint', str(STUDENT_CHECKPOINT),
    '--wham-repo', str(WHAM_REPO),
    '--wham-checkpoint', str(WHAM_CHECKPOINT),
    '--yolov8-weights', str(YOLOV8_WEIGHTS),
    '--smpl-model-directory', str(SMPL_MODEL_DIR),
    '--h36m-joint-regressor', str(H36M_REGRESSOR),
    '--sequences', str(SEQUENCES),
    '--frames', str(FRAMES_PER_SEQUENCE),
    '--pose-batch-size', str(POSE_BATCH_SIZE),
    '--student-batch-size', str(STUDENT_BATCH_SIZE),
    '--smpl-batch-size', str(SMPL_BATCH_SIZE),
    '--output', str(REPORT),
    '--per-sequence-output', str(PER_SEQUENCE),
]
environment = os.environ.copy()
environment['PYTHONPATH'] = str(SCRATCH_DIR) + os.pathsep + environment.get('PYTHONPATH', '')
print(' '.join(command), flush=True)
subprocess.run(command, check=True, env=environment)


In [ ]:
# Compact result table. These are the values needed for the report.
import json
from IPython.display import display
import pandas as pd

report = json.loads(REPORT.read_text())
rows = []
for name, variant in report['variants'].items():
    metrics = variant['metrics']
    rows.append({
        'variant': name,
        'recurrent frames': metrics['mpjpe_mm']['samples'],
        'PA-MPJPE (mm)': metrics['pa_mpjpe_mm']['mean'],
        'MPJPE (mm)': metrics['mpjpe_mm']['mean'],
        'PVE (mm)': metrics['pve_mm']['mean'],
        'Accel (official code)': metrics['accel_official_30fps']['mean'],
    })
mobile_reference = report['iphone_reference_paper_wham_single_person_subset']
reference_metrics = mobile_reference['metrics']
rows.append({
    'variant': 'paper_wham_bedlam_flip_single_person_reference',
    'recurrent frames': reference_metrics['mpjpe_mm']['samples'],
    'PA-MPJPE (mm)': reference_metrics['pa_mpjpe_mm']['mean'],
    'MPJPE (mm)': reference_metrics['mpjpe_mm']['mean'],
    'PVE (mm)': reference_metrics['pve_mm']['mean'],
    'Accel (official code)': reference_metrics['accel_official_30fps']['mean'],
})
display(pd.DataFrame(rows).round(3))
print('iPhone minus paper WHAM on the same single-person subset:', json.dumps(
    report['iphone_minus_paper_wham_same_single_person_subset'], indent=2
))
print('Baseline reproduction check:', json.dumps(
    report['baseline_reproduction'], indent=2
))
print('Controlled FastViT drift:', json.dumps(
    report['controlled_fastvit']['student_teacher_pose_drift']['all_joints_deg'],
    indent=2,
))
print('Mobile detection:', json.dumps(report['iphone_detection'], indent=2))


In [ ]:
# Export the exact phase-three FastViT as a clearly labelled diagnostic package.
# It is for the one-off phone latency run, not an acceptance claim.
COREML_PACKAGE = SCRATCH_DIR / 'FastViTNormalized.mlpackage'
export_command = [
    sys.executable, '-u', str(EXPORTER),
    '--weights', str(STUDENT_CHECKPOINT),
    '--output', str(COREML_PACKAGE),
    '--accept-product-validation',
    '--full-evaluation-report', str(REPORT),
]
subprocess.run(export_command, check=True, env=environment)
coreml_zip_base = OUTPUT_DIR / 'FastViTNormalized_phase3_diagnostic'
coreml_zip = Path(shutil.make_archive(
    str(coreml_zip_base), 'zip',
    root_dir=COREML_PACKAGE.parent, base_dir=COREML_PACKAGE.name,
))
manifest = {
    'student_checkpoint_sha256': EXPECTED_STUDENT_SHA256,
    'coreml_zip': coreml_zip.name,
    'deployment_accepted': False,
    'purpose': 'single physical-iPhone latency diagnostic',
    'full_evaluation_report': REPORT.name,
}
MANIFEST = OUTPUT_DIR / 'artifact_manifest.json'
MANIFEST.write_text(json.dumps(manifest, indent=2) + '\n')
print(json.dumps(manifest, indent=2))


In [ ]:
# One small download bundle; raw logs and temporary repositories stay out of it.
bundle_base = Path('/kaggle/working/wham_full_pipeline_tradeoff_results')
bundle = Path(shutil.make_archive(
    str(bundle_base), 'zip', root_dir=OUTPUT_DIR
))
print(f'Download: {bundle}')
print('Inside:', sorted(path.name for path in OUTPUT_DIR.iterdir()))
